In [1]:
import pandas as pd

df = pd.read_csv('data/thai_wikipron_5-4-2026.csv')
df

,writing,phonetic,count
0,ก,kɔː˧,2
1,ก,kɔː˧.kaj˨˩,2
2,ก.,kɔː˧,1
3,ก.ค.,kɔː˧.kʰɔː˧,1
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1
...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2
18307,ไฮ้,haj˦˥,1
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1


In [2]:
import re

def normalize_phonetic(phonetic: str) -> str:
    phonetic = (
        phonetic
        .replace('p̚', 'p')
        .replace('t̚', 't')
        .replace('k̚', 'k')
        .replace('a̯', 'ə')
        .replace('˥˩', '˦˩')
        .replace('˩˩˦', '˨˥')
    )

    SHORT_VOWELS = ['a', 'i', 'ɯ', 'u', 'e', 'ɤ', 'o', 'ɛ', 'ɔ']

    pattern = (
        '(' + '|'.join(map(re.escape, SHORT_VOWELS)) + ')'
        r'([˥˦˧˨˩]+)(?=\.|$)'
    )

    phonetic = re.sub(pattern, r'\1ʔ\2', phonetic)

    phonetic = re.sub(r'[.…]+', '.', phonetic)

    phonetic = re.sub(r'^\.|\.$', '', phonetic)

    return phonetic

print(normalize_phonetic('….daj˧.….nɯŋ˨˩'))

daj˧.nɯŋ˨˩


In [3]:
df['normalized_phonetic'] = df['phonetic'].apply(normalize_phonetic)
df

,writing,phonetic,count,normalized_phonetic
0,ก,kɔː˧,2,kɔː˧
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩
2,ก.,kɔː˧,1,kɔː˧
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧
...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩


In [4]:
import thai_gpa

def evaluate(text: str, ipa: str) -> tuple:
    result = thai_gpa.align(text, ipa)
    reconstructed_text = ''.join(s.reconstruct_text() for s in result)
    reconstructed_ipa = '.'.join(s.get_ipa(is_reduplicated=s.is_reduplicated) for s in result)
    return reconstructed_text, reconstructed_ipa

print(evaluate('การขัดกันของผลประโยชน์', 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩'))

('การขัดกันของผลประโยชน์', 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩')


In [5]:
from tqdm.auto import tqdm
tqdm.pandas()

def apply_evaluate(row):
    try:
        reconstructed_text, phonetic_answer = evaluate(row['writing'], row['normalized_phonetic'])
    except Exception as e:
        reconstructed_text, phonetic_answer = None, f'ERROR: {e}'
    return pd.Series([reconstructed_text, phonetic_answer])

df[['reconstructed_text', 'phonetic_answer']] = df.progress_apply(apply_evaluate, axis=1)
df.to_csv('data/test.csv', index=False)
df

C:\Users\pawi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 91%|█████████ | 16688/18310 [03:59<00:14, 111.21it/s]c:\Storage\repos\thai_grapheme_sandbox\thai_ipa.py:109: UserWarning: Warning: No explicit glottal stop at "e"
  warnings.warn(f'Warning: No explicit glottal stop at "{original}"')
100%|██████████| 18310/18310 [04:17<00:00, 71.18it/s] 


,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥,ไฮ้,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,None,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


In [6]:
mask = ~df["phonetic_answer"].str.startswith("ERROR:")
mismatches = df.loc[
    mask & (df["normalized_phonetic"] != df["phonetic_answer"]),
    ["writing", "normalized_phonetic", "phonetic_answer"]
]

mismatches

,writing,normalized_phonetic,phonetic_answer


In [7]:
errors = df[df["phonetic_answer"].str.startswith("ERROR:")]
errors

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18262,ไอซ์แลนด์,ʔajs˦˥.lɛːn˧,1,ʔajs˦˥.lɛːn˧,None,ERROR: argument of type 'NoneType' is not iter...
18263,ไอดอล,ʔaj˧.dɔl˥˩,2,ʔaj˧.dɔl˦˩,None,ERROR: argument of type 'NoneType' is not iter...
18278,ไอศครีม,ʔajs˧.kʰriːm˧,2,ʔajs˧.kʰriːm˧,None,ERROR: argument of type 'NoneType' is not iter...
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,None,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


In [8]:
errors.sample(10)   

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
2789,คอร์นวอลล์,kʰɔːn˧.wɔːl˧,1,kʰɔːn˧.wɔːl˧,None,ERROR: argument of type 'NoneType' is not iter...
8045,พรอวิเดนซ์,pʰrɔː˧.wi˦˥.dens˦˥,2,pʰrɔː˧.wiʔ˦˥.dens˦˥,None,ERROR: argument of type 'NoneType' is not iter...
13412,ออสเตรีย,ʔɔːs˦˥.tria̯˧,1,ʔɔːs˦˥.triə˧,None,ERROR: argument of type 'NoneType' is not iter...
9675,รบ.,rɔː˧.bɔː˧,1,rɔː˧.bɔː˧,None,ERROR: Could not align 'รบ.' with 'rɔː˧.bɔː˧'
6,ก.ย.,kɔː˧.jɔː˧,1,kɔː˧.jɔː˧,None,ERROR: Could not align 'ก.ย.' with 'kɔː˧.jɔː˧'
6464,บรัสเซลส์,bras˦˥.seːl˧,1,bras˦˥.seːl˧,None,ERROR: argument of type 'NoneType' is not iter...
10052,รีโมตคอนโทรล,riː˧.moːt̚˨˩.kʰɔn˧.tʰroːn˧,2,riː˧.moːt˨˩.kʰɔn˧.tʰroːn˧,None,ERROR: Could not align 'รีโมตคอนโทรล' with 'ri...
12283,สุโขทัย,suk̚˨˩.kʰoː˩˩˦.tʰaj˧,2,suk˨˩.kʰoː˨˥.tʰaj˧,None,ERROR: Could not align 'สุโขทัย' with 'suk˨˩.k...
5206,ทรัสต์,tʰrat̚˦˥,2,tʰrat˦˥,None,ERROR: Could not align 'ทรัสต์' with 'tʰrat˦˥'
10894,วิทยาการหุ่นยนต์,wit̚˦˥.tʰa˦˥.jaː˧.kaːn˧.hun˨˩.jon˧,1,wit˦˥.tʰaʔ˦˥.jaː˧.kaːn˧.hun˨˩.jon˧,None,ERROR: Could not align 'วิทยาการหุ่นยนต์' with...


In [9]:
mask = ~errors["normalized_phonetic"].str.contains(
    r"[lsf][˥˦˧˨˩]", regex=True, na=False
)
filtered_errors = errors[mask]
filtered_errors

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
17938,ไซบูทรามีน,saj˧.buː˧.tʰraː˧.miːn˧,1,saj˧.buː˧.tʰraː˧.miːn˧,None,ERROR: Could not align 'ไซบูทรามีน' with 'saj˧...
18041,ไปรษณียบัตร,praj˧.sa˨˩.niː˧.bat̚˨˩,2,praj˧.saʔ˨˩.niː˧.bat˨˩,None,ERROR: Could not align 'ไปรษณียบัตร' with 'pra...
18044,ไปรษณีย์อิเล็กทรอนิกส์,praj˧.sa˨˩.niː˧.ʔi˨˩.lek̚˦˥.tʰrɔː˧.nik̚˨˩,1,praj˧.saʔ˨˩.niː˧.ʔiʔ˨˩.lek˦˥.tʰrɔː˧.nik˨˩,None,ERROR: Could not align 'ไปรษณีย์อิเล็กทรอนิกส์...
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,None,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


In [10]:
filtered_errors.sample(10)

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
9678,รปภ.,rɔː˧.pɔː˧.pʰɔː˧,1,rɔː˧.pɔː˧.pʰɔː˧,None,ERROR: Could not align 'รปภ.' with 'rɔː˧.pɔː˧....
7959,พ,pʰɔː˧.pʰaːn˧,2,pʰɔː˧.pʰaːn˧,None,ERROR: Could not align 'พ' with 'pʰɔː˧.pʰaːn˧'
6403,บดินทร,bɔː˧.din˧.tʰra˦˥.,1,bɔː˧.din˧.tʰraʔ˦˥,None,ERROR: Could not align 'บดินทร' with 'bɔː˧.din...
8,กกต.,kɔː˧.kɔː˧.tɔː˧,1,kɔː˧.kɔː˧.tɔː˧,None,ERROR: Could not align 'กกต.' with 'kɔː˧.kɔː˧....
6512,บวร,bɔː˧.wɔː˧.ra˦˥.,3,bɔː˧.wɔː˧.raʔ˦˥,None,ERROR: Could not align 'บวร' with 'bɔː˧.wɔː˧.r...
1686,ก็ได้,kɔː˥˩.daːj˥˩,1,kɔː˦˩.daːj˦˩,None,ERROR: Could not align 'ก็ได้' with 'kɔː˦˩.daː...
13309,อยู่ไฟ,juː˨˩.faj˧,1,juː˨˩.faj˧,None,ERROR: Could not align 'อยู่ไฟ' with 'juː˨˩.faj˧'
15823,เล่าฦๅ,law˥˩.lɯː˧,1,law˦˩.lɯː˧,None,ERROR: Could not align 'เล่าฦๅ' with 'law˦˩.lɯː˧'
1388,การเสียดาย,kʰwaːm˧.sia̯˩˩˦.daːj˧,1,kʰwaːm˧.siə˨˥.daːj˧,None,ERROR: Could not align 'การเสียดาย' with 'kʰwa...
14265,เกียรติ์,kia̯n˧,1,kiən˧,None,ERROR: Could not align 'เกียรติ์' with 'kiən˧'
